# Análisis de Tabu Search — Job Shop Scheduling

**Objetivo:** minimización del makespan (Cmax)  
**Algoritmos comparados:** TS-Básico · TS-Intermedio · TS-Avanzado  
**Instancias TSPLIB:** 5 instancias de referencia  


In [ ]:
import os, json, glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

BASE = os.path.abspath('..')   # run from notebooks/
RESULTS = os.path.join(BASE, 'results')
OUTPUT  = os.path.join(BASE, 'output')


In [ ]:
# Load per-problem CSV
df_all = pd.read_csv(os.path.join(RESULTS, 'tabla_consolidada.csv'))
prob_label = 'Jobshop'
df = df_all[df_all['Problema'].str.lower() == df_all['Problema'].str.lower().map(lambda x: x if 'jobshop' in x.lower() else '')].copy()
# Simpler filter:
df = pd.read_csv(os.path.join(RESULTS, 'tabla_jobshop.csv'))
df.head(10)


## 1. Tabla de GAP (%) por instancia


In [ ]:
gap_cols = ['Instancia','BKS','TS_Basico','GAP_B(%)','TS_Intermedio','GAP_I(%)','TS_Avanzado','GAP_A(%)']
print(df[gap_cols].to_string(index=False))


## 2. GAP (%) por instancia y algoritmo


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(df))
w = 0.25
ax.bar(x - w, df['GAP_B(%)'].astype(float), w, label='TS-Básico',      color='#4c72b0')
ax.bar(x,     df['GAP_I(%)'].astype(float), w, label='TS-Intermedio',  color='#dd8452')
ax.bar(x + w, df['GAP_A(%)'].astype(float), w, label='TS-Avanzado',    color='#55a868')
ax.set_xticks(x)
ax.set_xticklabels(df['Instancia'], rotation=30, ha='right')
ax.set_ylabel('GAP (%)')
ax.set_title('GAP respecto al BKS — Job Shop Scheduling')
ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
ax.legend()
plt.tight_layout()
plt.savefig(f'../paper/figuras/gap_{prob}.pdf', bbox_inches='tight')
plt.show()


## 3. Curvas de convergencia


In [ ]:
ALGOS = ['ts_basico', 'ts_intermedio', 'ts_avanzado']
ALGO_LABELS = {'ts_basico':'TS-Básico','ts_intermedio':'TS-Intermedio','ts_avanzado':'TS-Avanzado'}
COLORS = {'ts_basico':'#4c72b0','ts_intermedio':'#dd8452','ts_avanzado':'#55a868'}

instances = df['Instancia'].tolist()
n_inst = len(instances)
fig, axes = plt.subplots(1, n_inst, figsize=(4*n_inst, 4), sharey=False)
if n_inst == 1: axes = [axes]

for ax, inst in zip(axes, instances):
    for algo in ALGOS:
        # Find result folder
        pattern = os.path.join(OUTPUT, 'jobshop', f'{inst}_{algo}_*')
        folders = sorted(glob.glob(pattern))
        if not folders: continue
        with open(os.path.join(folders[-1], 'resultado.json')) as f:
            res = json.load(f)
        hist = res.get('history', [])
        if hist:
            iters = [h[0] for h in hist]
            vals  = [h[1] for h in hist]
            ax.plot(iters, vals, label=ALGO_LABELS[algo], color=COLORS[algo], linewidth=1.5)
    ax.set_title(inst, fontsize=10)
    ax.set_xlabel('Iteración')
    ax.set_ylabel(f'{metrica}')
    ax.legend(fontsize=8)

fig.suptitle('Convergencia — Job Shop Scheduling', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'../paper/figuras/convergencia_{prob}.pdf', bbox_inches='tight')
plt.show()


## 4. Tiempo de ejecución (segundos)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(df))
w = 0.25
ax.bar(x - w, df['T_B(s)'].astype(float), w, label='TS-Básico',      color='#4c72b0')
ax.bar(x,     df['T_I(s)'].astype(float), w, label='TS-Intermedio',  color='#dd8452')
ax.bar(x + w, df['T_A(s)'].astype(float), w, label='TS-Avanzado',    color='#55a868')
ax.set_xticks(x)
ax.set_xticklabels(df['Instancia'], rotation=30, ha='right')
ax.set_ylabel('Tiempo (s)')
ax.set_title('Tiempo de ejecución — Job Shop Scheduling')
ax.legend()
plt.tight_layout()
plt.savefig(f'../paper/figuras/tiempo_{prob}.pdf', bbox_inches='tight')
plt.show()


## 5. Resumen estadístico


In [ ]:
summary = pd.DataFrame({
    'Algoritmo': ['TS-Básico','TS-Intermedio','TS-Avanzado'],
    'GAP Promedio (%)': [
        df['GAP_B(%)'].astype(float).mean(),
        df['GAP_I(%)'].astype(float).mean(),
        df['GAP_A(%)'].astype(float).mean(),
    ],
    'GAP Mínimo (%)': [
        df['GAP_B(%)'].astype(float).min(),
        df['GAP_I(%)'].astype(float).min(),
        df['GAP_A(%)'].astype(float).min(),
    ],
    'GAP Máximo (%)': [
        df['GAP_B(%)'].astype(float).max(),
        df['GAP_I(%)'].astype(float).max(),
        df['GAP_A(%)'].astype(float).max(),
    ],
    'Tiempo Prom. (s)': [
        df['T_B(s)'].astype(float).mean(),
        df['T_I(s)'].astype(float).mean(),
        df['T_A(s)'].astype(float).mean(),
    ],
})
summary = summary.set_index('Algoritmo')
print(summary.to_string(float_format=lambda x: f'{x:.4f}'))
